In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import welch


In [ ]:

# 🔹 Define Frequency Bands
bands = {
    "Delta": (0.5, 4),
    "Theta": (4, 8),
    "Alpha": (8, 13),
    "Beta": (13, 30),
    "Gamma": (30, 50),
}


In [ ]:

def compute_band_power(window, fs=256):
    """Compute Band Power for all EEG channels in a given window."""
    band_power_features = []
    
    for i in range(window.shape[1]):  # Iterate over EEG channels
        f, Pxx = welch(window[:, i], fs=fs, nperseg=fs, window='hann', scaling='density')
        
        # Compute power for each frequency band
        channel_band_powers = []
        for band, (low, high) in bands.items():
            band_mask = (f >= low) & (f <= high)
            power = np.trapz(Pxx[band_mask], f[band_mask])  # Compute power using integration
            channel_band_powers.append(power)
        
        band_power_features.extend(channel_band_powers)  # Flatten into a single list
    
    return band_power_features


In [ ]:

def sliding_window_band_power(eeg_data, outcomes, eeg_columns, window_size, step_size, fs=256):
    """Extract Band Power features using a sliding window approach."""
    all_band_power_features = []
    targets = []
    n_samples = eeg_data.shape[0]
    
    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        window = eeg_data[start:end]
        outcome_window = outcomes[start:end]

        # Compute Band Power features for this window
        band_power_values = compute_band_power(window, fs)  # Shape: (num_channels * num_bands)

        all_band_power_features.append(band_power_values)
        targets.append(1 if np.any(outcome_window) else 0)  # Assign target based on Outcome
    
    return np.array(all_band_power_features), np.array(targets)


In [ ]:

# Main processing
if __name__ == "__main__":
    # Load EEG data
    eeg_data_path = '/Users/puchku-home/Study/PROJECT/EEG/EEG Assets/chbmit_preprocessed_data.csv' 
    data = pd.read_csv(eeg_data_path)
    eeg_columns = [col for col in data.columns if col != 'Outcome']

    # Convert to NumPy arrays
    eeg_data = np.asarray(data[eeg_columns].values, dtype=np.float32)
    outcomes = np.asarray(data['Outcome'].values, dtype=np.float32)

    # Windowing parameters
    fs = 256  # Sampling frequency
    window_size = fs * 1  # 1-second windows
    step_size = window_size // 2  # 50% overlap

    # Compute Band Power features
    band_power_features, targets = sliding_window_band_power(eeg_data, outcomes, eeg_columns, window_size, step_size, fs)

    # Convert to DataFrame and save
    num_bands = len(bands)
    band_feature_names = [f"{col}_{band}" for col in eeg_columns for band in bands.keys()]
    
    band_power_df = pd.DataFrame(band_power_features, columns=band_feature_names)
    band_power_df['target'] = targets
    
    output_file_path = '/Users/puchku-home/Downloads/Frequency Feature  Generalised/Band_Power_Results-1.csv'
    band_power_df.to_csv(output_file_path, index=False)
    
    print(f"✅ Band Power feature extraction complete. Features saved to '{output_file_path}'")
